# Fireworks LLM Evaluation
This notebook loads `.env`, reads tasks/truth/results, and uses a Fireworks model as an LLM judge.

In [1]:
%pip install -q openai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os, json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key=os.getenv("FIREWORKS_API_KEY")
base_url="https://api.fireworks.ai/inference/v1"
model=os.getenv("EVAL_MODEL") or os.getenv("MODEL") or os.getenv("ALLOWED_MODELS","accounts/fireworks/models/llama-v3p1-8b-instruct")

if "," in model:
    model=model.split(",")[0].strip()

client=OpenAI(api_key=api_key,base_url=base_url)

ROOT=Path("..")
OUT=ROOT/"output"
IN=ROOT/"input"
with open(IN/"tasks.json","r",encoding="utf-8") as f:
    tasks={x["task_id"]:x["prompt"] for x in json.load(f)}
with open(OUT/"truth.json","r",encoding="utf-8") as f:
    truth={x["task_id"]:x["answer"] for x in json.load(f)}
with open(OUT/"results.json","r",encoding="utf-8") as f:
    results={x["task_id"]:x["answer"] for x in json.load(f)}

print("Using model:",model)
print("Loaded",len(tasks),"tasks")


Using model: accounts/fireworks/models/minimax-m3
Loaded 19 tasks


In [2]:
import json
import re

def judge(prompt, gt, pred):
    eval_prompt = f"""
You are an impartial evaluator.

Task:
{prompt}

Reference Answer:
{gt}

Student Answer:
{pred}

Score the student's answer from 0 to 100.

Return ONLY valid JSON.

{{
    "score": 95,
    "reason": "Brief explanation."
}}
"""

    r = client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=512,
        messages=[
            {
                "role": "user",
                "content": eval_prompt
            }
        ]
    )

    # Print the raw response for debugging
    print("=" * 80)
    print(r.model_dump_json(indent=2))
    print("=" * 80)

    msg = r.choices[0].message

    # Use content first, fall back to reasoning_content
    content = msg.content

    if content is None:
        content = getattr(msg, "reasoning_content", None)

    if content is None:
        raise RuntimeError(
            "The model returned neither content nor reasoning_content."
        )

    # Extract the first JSON object from the response
    match = re.search(r"\{.*\}", content, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON found in:\n{content}")

    return json.loads(match.group(0))

In [3]:
print("Base URL:", base_url)
print("Model:", model)

rows = []

for tid, prompt in tasks.items():
    print(f"Evaluating {tid}...")

    ev = judge(prompt, truth[tid], results.get(tid, ""))

    rows.append({
        "task_id": tid,
        "score": ev["score"],
        "reason": ev["reason"]
    })

df = pd.DataFrame(rows)
display(df)

print("Average Score:", df["score"].mean())

df.to_csv("evaluation.csv", index=False)
print("Saved evaluation.csv")


Base URL: https://api.fireworks.ai/inference/v1
Model: accounts/fireworks/models/minimax-m3
Evaluating test-01...
{
  "id": "chatcmpl-2d7645bb16c24d5d981377dd9d929d64",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\n    \"score\": 100,\n    \"reason\": \"The student correctly identifies Jupiter as the largest planet and accurately states its main composition as hydrogen and helium. The additional details about percentages and comparison to the Sun are accurate and enhance the answer.\"\n}",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning_content": "The student correctly identifies Jupiter as the largest planet and correctly states its main composition as hydrogen and helium. They even provide additional accurate details about the percentages and comparison to the S

,task_id,score,reason
0,test-01,100,The student correctly identifies Jupiter as th...
1,test-02,100,The student's answer of 275 matches the refere...
2,test-03,95,The student's answer 'Mixed' correctly identif...
3,test-04,92,The student's answer is a single sentence that...
4,test-05,100,The student's answer perfectly matches the ref...
5,test-06,90,The student correctly fixes the bug by adding ...
6,test-07,100,The student's answer 'Charlie' correctly ident...
7,test-08,95,The student's answer is correct and more robus...
8,test-09,100,The student's answer is exactly the same as th...
9,test-10,92,"The student's answer is professional, removes ..."


Average Score: 92.57894736842105
Saved evaluation.csv
